# Construir el catálogo STAC del atlas (`data/stac/`)

Multi-año: descubre los años con productos en `data/raw/UTCI/<año>/`,
genera un COG por año (10 bandas, una por nivel de estrés) y arma el
catálogo: colección `utci` con items `utci-hourly-<año>` y
`utci-levels-<año>` (assets NetCDF + COG). El STAC cataloga, no duplica.

Si `data/stac/catalog.json` ya existe, se conserva y sólo se reemplaza la
colección `utci` (las demás colecciones, p. ej. `inegi`, quedan intactas).


In [1]:
from datetime import datetime, timezone
from pathlib import Path

import pystac
import rioxarray  # noqa: F401  (registra el accessor .rio)
import xarray as xr

ROOT = Path("..").resolve()                      # repo (libreta en notebooks/)
RAW_UTCI = ROOT / "data" / "raw" / "UTCI"
STAC_DIR = ROOT / "data" / "stac"

NC_MEDIA_TYPE = "application/x-netcdf"
COG_MEDIA_TYPE = "image/tiff; application=geotiff; profile=cloud-optimized"

In [2]:
years = sorted(
    int(p.parent.name)
    for p in RAW_UTCI.glob("*/UTCI_levels_*.nc")
    if p.parent.name.isdigit()
)
print("Años encontrados:", years)

Años encontrados: [2022, 2023]


In [3]:
def build_cog(year: int) -> Path:
    """NetCDF de niveles -> COG multibanda EPSG:4326. Regenera si está viejo."""
    nc = RAW_UTCI / str(year) / f"UTCI_levels_{year}.nc"
    cog = RAW_UTCI / str(year) / f"UTCI_levels_{year}_cog.tif"
    if cog.exists() and cog.stat().st_mtime >= nc.stat().st_mtime:
        print(f"{cog.name}: al día")
        return cog
    lv = xr.open_dataset(nc)
    da = (
        lv["hours"].astype("int32")
        .rename({"lat": "y", "lon": "x"})
        .transpose("level", "y", "x")
        .rio.write_crs("EPSG:4326")
    )
    # Descripciones de banda = nombres de los niveles
    da.attrs["long_name"] = lv["level"].attrs["names_es"].split("; ")
    da.rio.to_raster(cog, driver="COG", compress="DEFLATE")
    lv.close()
    print(f"{cog.name}: generado ({cog.stat().st_size / 1e3:.0f} kB)")
    return cog

cogs = {y: build_cog(y) for y in years}

UTCI_levels_2022_cog.tif: al día
UTCI_levels_2023_cog.tif: al día


In [4]:
meta = {}
for y in years:
    lv = xr.open_dataset(RAW_UTCI / str(y) / f"UTCI_levels_{y}.nc")
    meta[y] = {
        "bbox": [
            float(lv.lon.min()), float(lv.lat.min()),
            float(lv.lon.max()), float(lv.lat.max()),
        ],
        "levels": lv.level.attrs["flag_meanings"].split(),
        "levels_es": lv.level.attrs["names_es"].split("; "),
        "bounds_degC": [float(b) for b in lv.attrs["utci_bounds_degC"]],
    }
    lv.close()
    print(y, meta[y]["bbox"])

# bbox de la colección: unión de todos los años
union = [
    min(m["bbox"][0] for m in meta.values()),
    min(m["bbox"][1] for m in meta.values()),
    max(m["bbox"][2] for m in meta.values()),
    max(m["bbox"][3] for m in meta.values()),
]

2022 [-118.5, 14.0, -86.0, 33.0]
2023 [-118.5, 14.0, -86.0, 33.5]


In [5]:
# Si el catálogo ya existe (p. ej. con la colección `inegi` de la libreta 007),
# se reutiliza y sólo se reemplaza la colección `utci`; así cada libreta es
# dueña de su colección y ninguna borra las demás.
if (STAC_DIR / "catalog.json").exists():
    catalog = pystac.Catalog.from_file(str(STAC_DIR / "catalog.json"))
    if catalog.get_child("utci") is not None:
        catalog.remove_child("utci")
else:
    catalog = pystac.Catalog(
        id="atlas",
        description=(
            "Atlas de confort térmico de México (IER-UNAM). "
            "Catálogo de datos rasterizados para la webapp."
        ),
        title="Atlas IER-UNAM",
    )

collection = pystac.Collection(
    id="utci",
    title="UTCI — Universal Thermal Climate Index (ERA5-HEAT)",
    description=(
        "UTCI horario ERA5-HEAT (ECMWF, v1.1) recortado a México y productos "
        "derivados: horas anuales por nivel de estrés térmico (escala ISB, "
        "10 niveles). Malla regular de 0.25°."
    ),
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([union]),
        temporal=pystac.TemporalExtent([[
            datetime(years[0], 1, 1, tzinfo=timezone.utc),
            datetime(years[-1], 12, 31, 23, tzinfo=timezone.utc),
        ]]),
    ),
    license="CC-BY-4.0",
    keywords=["UTCI", "confort térmico", "ERA5-HEAT", "México"],
)
catalog.add_child(collection)

<Link rel=child target=<Collection id=utci>>

In [6]:
def bbox_geometry(bbox):
    return {
        "type": "Polygon",
        "coordinates": [[
            [bbox[0], bbox[1]], [bbox[2], bbox[1]],
            [bbox[2], bbox[3]], [bbox[0], bbox[3]],
            [bbox[0], bbox[1]],
        ]],
    }

for y in years:
    m = meta[y]
    t0 = datetime(y, 1, 1, tzinfo=timezone.utc)
    t1 = datetime(y, 12, 31, 23, tzinfo=timezone.utc)
    tiempo = {
        "start_datetime": t0.isoformat(),
        "end_datetime": t1.isoformat(),
    }

    hourly = pystac.Item(
        id=f"utci-hourly-{y}",
        geometry=bbox_geometry(m["bbox"]),
        bbox=m["bbox"],
        datetime=None,
        properties={
            **tiempo,
            "utci:variable": "utci",
            "utci:units": "degC",
        },
    )
    hourly.add_asset(
        "data",
        pystac.Asset(
            href=str(RAW_UTCI / str(y) / f"UTCI_Mexico_{y}.nc"),
            media_type=NC_MEDIA_TYPE,
            title=f"UTCI horario México {y} (NetCDF)",
            roles=["data"],
        ),
    )
    collection.add_item(hourly)

    levels = pystac.Item(
        id=f"utci-levels-{y}",
        geometry=bbox_geometry(m["bbox"]),
        bbox=m["bbox"],
        datetime=None,
        properties={
            **tiempo,
            "utci:variable": "hours",
            "utci:units": "h",
            "utci:levels": m["levels"],
            "utci:levels_es": m["levels_es"],
            "utci:bounds_degC": m["bounds_degC"],
        },
    )
    levels.add_asset(
        "data",
        pystac.Asset(
            href=str(RAW_UTCI / str(y) / f"UTCI_levels_{y}.nc"),
            media_type=NC_MEDIA_TYPE,
            title=f"Horas anuales por nivel de estrés UTCI, México {y} (NetCDF)",
            roles=["data"],
        ),
    )
    levels.add_asset(
        "cog",
        pystac.Asset(
            href=str(cogs[y]),
            media_type=COG_MEDIA_TYPE,
            title=f"Horas anuales por nivel de estrés UTCI, México {y} (COG, banda = nivel+1)",
            roles=["data", "overview"],
        ),
    )
    collection.add_item(levels)

In [7]:
catalog.normalize_hrefs(str(STAC_DIR))
catalog.make_all_asset_hrefs_relative()
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
print("Guardado en", STAC_DIR)
catalog.describe()

Guardado en /Users/gbv/atlas/data/stac
* <Catalog id=atlas>
    * <Collection id=inegi>
      * <Item id=medi-ageb-2020>
      * <Item id=medi-mun-2020>
      * <Item id=medi-grid-2020>
    * <Collection id=utci>
      * <Item id=utci-hourly-2022>
      * <Item id=utci-levels-2022>
      * <Item id=utci-hourly-2023>
      * <Item id=utci-levels-2023>


In [8]:
for obj in [catalog, collection, *collection.get_items()]:
    obj.validate()
    print("OK", obj.id)

OK atlas
OK utci
OK utci-hourly-2022
OK utci-levels-2022
OK utci-hourly-2023
OK utci-levels-2023


In [9]:
cat = pystac.Catalog.from_file(str(STAC_DIR / "catalog.json"))
col = cat.get_child("utci")
for y in years:
    it = col.get_item(f"utci-levels-{y}")
    nc_path = Path(it.assets["data"].get_absolute_href())
    ds = xr.open_dataset(nc_path)
    cog = xr.open_dataarray(Path(it.assets["cog"].get_absolute_href()), engine="rasterio")
    igual = int(ds["hours"].sel(level=7).sum()) == int(cog.sel(band=8).sum())
    print(y, "nc==cog (nivel 7):", igual, "| máx:", int(ds["hours"].sel(level=7).max()), "h")
    ds.close()
    cog.close()

2022 nc==cog (nivel 7): True | máx: 3165 h
2023 nc==cog (nivel 7): True | máx: 3761 h
